# Notebook 03 – Goal 3: Unified Evaluation Framework

**Authors:** Ivo Rambaldi & Tommaso Petrelli  
**Course:** Ethics for Artificial Intelligence, University of Bologna, a.y. 2025–2026

---

## Goal 3 — Utility and Fairness Measurement Framework

This notebook implements Goal 3 of the project:

> Implement a unified evaluation module computing all utility and fairness metrics for any (train-data, test-data, classifier) triple. Validate metric implementations on the real-data baseline.

### Structure

1. **Setup** — imports, config, constants  
2. **Data loading** — reconstruct `DataSplit` from Goal 1 outputs; binarise protected attributes; encode features  
3. **Synthetic data loading** — load Goal 2 outputs; align feature schema  
4. **Real-data baseline validation** — run `Evaluator.baseline()` for all three classifiers  
5. **TSTR evaluation loop** — full `(generator × classifier)` matrix  
6. **Utility analysis** — tables and bar charts of BA / F1 / AUC and MMD  
7. **Fairness analysis** — DPD / EOD / DI tables, per-attribute plots, delta heatmaps  
8. **Save results**

## 1 · Setup

In [1]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import OrdinalEncoder, StandardScaler

sys.path.append(str(Path.cwd().parent))
warnings.filterwarnings("ignore")

from src.utils import get_logger, load_config
from src.data.preprocessor import DataSplit, save_split
from src.models.classifiers import build_classifier, CLASSIFIER_DISPLAY_NAMES
from src.evaluation.evaluator import Evaluator
from src.utils import (
    plot_utility_bar,
    plot_fairness_bar,
    plot_mmd_bar,
    plot_delta_heatmap,
    plot_utility_fairness_scatter,
    plot_per_attribute_fairness,
    plot_class_balance,
)

logger = get_logger("notebook_03")

In [2]:
cfg = load_config()

TARGET_COL  = cfg["target"]["column_name"]            # "target_high_perf"
SEED        = ["seed"]
CLASSIFIERS = list(CLASSIFIER_DISPLAY_NAMES.keys())

METHODS     = list(cfg["generation"]["output_subdirs"].keys()) # [ctgan, tvae, ...]

BASELINE    =  "real"

# Protected attributes we construct in this notebook (must be binary)
PROTECTED_ATTRS = ["s_gender", "f_ESCS_binary"]

print(f"Target       : {TARGET_COL}")
print(f"Classifiers  : {CLASSIFIERS}")
print(f"Gen. methods : {METHODS}")
print(f"Protected    : {PROTECTED_ATTRS}")
print(f"Seed         : {SEED}")

Target       : target_high_perf
Classifiers  : ['logistic_regression', 'xgboost', 'mlp']
Gen. methods : ['ctgan', 'tvae', 'gaussian_copula', 'smote']
Protected    : ['s_gender', 'f_ESCS_binary']
Seed         : ['seed']


## 2 · Data Loading & DataSplit Construction

We load the train / test CSVs produced by Notebook 01, binarise the protected
attributes, apply a consistent OrdinalEncoder + StandardScaler (fit on the real
training set only), and wrap everything in a `DataSplit` that the `Evaluator`
expects.

### Protected attribute binarisation

| Attribute | Raw column | Binarisation |
|-----------|-----------|--------------|
| Gender | `s_gender` | Used as-is (2 categories after recoding) |
| Socioeconomic status | `f_ESCS` | `f_ESCS >= 0` → `"advantaged"` / `"disadvantaged"` (0 ≈ OECD mean) |

Rows where `f_ESCS` is NaN receive the mode category so that `_check_binary_sensitive`
does not encounter a third value. The NaN indicator column `f_ESCS_was_nan` is still
present in `X` and is used by the classifiers normally.

In [3]:
train_df = pd.read_csv(cfg["paths"]["train_data"])
test_df  = pd.read_csv(cfg["paths"]["test_data"])

print(f"Train : {train_df.shape}   Test : {test_df.shape}")
print(f"\nClass balance (train):\n{train_df[TARGET_COL].value_counts(normalize=True).round(3)}")
print(f"\nClass balance (test):\n{test_df[TARGET_COL].value_counts(normalize=True).round(3)}")

Train : (58521, 304)   Test : (14631, 304)


KeyError: 'target_high_perf'

In [ ]:
def binarise_escs(series: pd.Series) -> pd.Series:
    """Binarise f_ESCS at 0 (OECD mean). NaN → mode of non-null values."""
    binarised = series.apply(
        lambda x: "advantaged" if pd.notna(x) and x >= 0 else ("disadvantaged" if pd.notna(x) else np.nan)
    )
    mode = binarised.mode(dropna=True)[0]
    return binarised.fillna(mode)


def build_protected(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build the protected-attribute DataFrame from a feature/target DataFrame.
    Returns a two-column DataFrame: s_gender and f_ESCS_binary.
    Raises a clear error if either source column is absent.
    """
    protected = {}

    # s_gender — already categorical; verify binary
    if "s_gender" in df.columns:
        g = df["s_gender"].astype(str).str.strip()
        n_vals = g.nunique(dropna=True)
        if n_vals != 2:
            print(f"  WARNING: s_gender has {n_vals} unique values (expected 2): {g.unique()}")
        protected["s_gender"] = g
    else:
        print("  WARNING: s_gender not found in DataFrame — skipping.")

    # f_ESCS_binary — binarised continuous ESCS
    if "f_ESCS" in df.columns:
        protected["f_ESCS_binary"] = binarise_escs(df["f_ESCS"])
    else:
        print("  WARNING: f_ESCS not found — skipping ESCS fairness.")

    return pd.DataFrame(protected, index=df.index)


protected_train = build_protected(train_df)
protected_test  = build_protected(test_df)

for attr in protected_train.columns:
    print(f"\n{attr}:")
    print(f"  Train: {protected_train[attr].value_counts().to_dict()}")
    print(f"  Test : {protected_test[attr].value_counts().to_dict()}")

In [ ]:
def make_feature_matrix(df: pd.DataFrame, target_col: str) -> pd.DataFrame:
    """Drop target + any leftover was_nan flag columns that shouldn't be features."""
    drop_cols = [target_col] + [c for c in df.columns if c.endswith("_was_nan")]
    return df.drop(columns=[c for c in drop_cols if c in df.columns])


X_train_raw = make_feature_matrix(train_df, TARGET_COL)
X_test_raw  = make_feature_matrix(test_df,  TARGET_COL)
y_train     = train_df[TARGET_COL].astype(int)
y_test      = test_df[TARGET_COL].astype(int)

print(f"X_train : {X_train_raw.shape}   X_test : {X_test_raw.shape}")
print(f"Positive rate — train: {y_train.mean():.1%}  test: {y_test.mean():.1%}")

### Feature encoding

Logistic Regression and MLP require fully numerical input; XGBoost also benefits
from it when using the sklearn wrapper. We apply:

- **OrdinalEncoder** (fit on X_train) to all `object`-dtype columns  
- **StandardScaler** (fit on X_train) to all numerical columns  

The fitted transformers are stored in `DataSplit.encoders` and `DataSplit.scaler`
so they can be applied to synthetic data in the same way.

In [ ]:
def fit_encoders(X_train: pd.DataFrame):
    """Fit OrdinalEncoder on categorical cols, StandardScaler on numerical."""
    cat_cols = X_train.select_dtypes(include="object").columns.tolist()
    num_cols = X_train.select_dtypes(include="number").columns.tolist()

    enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    scaler = StandardScaler()

    if cat_cols:
        enc.fit(X_train[cat_cols])
    if num_cols:
        scaler.fit(X_train[num_cols])

    return enc, scaler, cat_cols, num_cols


def apply_encoders(X: pd.DataFrame, enc, scaler, cat_cols, num_cols) -> pd.DataFrame:
    """Apply fitted OrdinalEncoder and StandardScaler. Returns a DataFrame."""
    X_out = X.copy()
    present_cat = [c for c in cat_cols if c in X_out.columns]
    present_num = [c for c in num_cols if c in X_out.columns]

    if present_cat:
        X_out[present_cat] = enc.transform(X_out[present_cat])
    if present_num:
        X_out[present_num] = scaler.transform(X_out[present_num])

    # Fill any remaining NaNs (sensitive cols not imputed in Goal 1) with 0
    X_out = X_out.fillna(0.0)
    return X_out.astype(float)


enc, scaler, cat_cols, num_cols = fit_encoders(X_train_raw)
X_train = apply_encoders(X_train_raw, enc, scaler, cat_cols, num_cols)
X_test  = apply_encoders(X_test_raw,  enc, scaler, cat_cols, num_cols)

print(f"Encoded X_train: {X_train.shape}  (cat={len(cat_cols)}, num={len(num_cols)})")
print(f"Any NaNs remaining: {X_train.isna().any().any()}")

In [ ]:
FEATURE_NAMES  = X_train.columns.tolist()
PROTECTED_COLS = [c for c in PROTECTED_ATTRS if c in protected_train.columns]

split = DataSplit(
    X_train        = X_train,
    X_test         = X_test,
    y_train        = y_train,
    y_test         = y_test,
    protected_train= protected_train[PROTECTED_COLS],
    protected_test = protected_test[PROTECTED_COLS],
    feature_names  = FEATURE_NAMES,
    target_name    = TARGET_COL,
    protected_attrs= PROTECTED_COLS,
    encoders       = {"ordinal": enc},
    scaler         = scaler,
)

# Persist for downstream notebooks (Goal 4 / 5)
save_split(split, cfg["paths"]["processed_data"].parent / "data_split.pkl")
print("DataSplit built and saved.")
print(f"  X_train      : {split.X_train.shape}")
print(f"  X_test       : {split.X_test.shape}")
print(f"  Protected    : {PROTECTED_COLS}")

## 3 · Load Synthetic Datasets (Goal 2 outputs)

We load the four CSV files produced by Notebook 02, align their column schema to
the real training set (inner join on feature names), and apply the same
OrdinalEncoder + StandardScaler fitted above.

> **Note:** SMOTE produces an oversampled dataset that includes all original
> training rows plus new synthetic minority-class rows, so its size will differ
> from the other generators.

In [ ]:
from src.utils.config import get_synthetic_output_path


def load_synthetic(method: str) -> tuple[pd.DataFrame, pd.Series]:
    """Load synthetic CSV, encode features, return (X_synth, y_synth)."""
    path = get_synthetic_output_path(cfg, method)
    if not path.exists():
        raise FileNotFoundError(
            f"Synthetic data not found at {path}.\n"
            f"Run Notebook 02 first to generate it."
        )
    df = pd.read_csv(path)

    if TARGET_COL not in df.columns:
        raise ValueError(f"Target column '{TARGET_COL}' missing from {path}.")

    y_synth = df[TARGET_COL].astype(int)

    # Keep only features present in the real training set
    feat_cols = [c for c in FEATURE_NAMES if c in df.columns]
    missing   = set(FEATURE_NAMES) - set(df.columns)
    if missing:
        print(f"  [{method}] WARNING: {len(missing)} feature cols absent "
              f"from synthetic data — filled with 0.")

    X_synth_raw = pd.DataFrame(index=df.index)
    for col in FEATURE_NAMES:
        X_synth_raw[col] = df[col] if col in df.columns else 0.0

    X_synth = apply_encoders(X_synth_raw, enc, scaler, cat_cols, num_cols)
    return X_synth, y_synth


synthetic_data: dict[str, tuple[pd.DataFrame, pd.Series]] = {}

for method in METHODS:
    try:
        X_s, y_s = load_synthetic(method)
        synthetic_data[method] = (X_s, y_s)
        pos_rate = y_s.mean()
        print(f"  [{method}]  rows={len(X_s):,}   +class={pos_rate:.1%}   "
              f"cols={X_s.shape[1]}")
    except FileNotFoundError as e:
        print(f"  [{method}]  SKIPPED — {e}")

print(f"\nLoaded {len(synthetic_data)}/{len(METHODS)} synthetic datasets.")

In [ ]:
# Quick sanity check: class balance across all datasets
raw_synthetics_for_plot = {}
for method, (X_s, y_s) in synthetic_data.items():
    df_plot = X_s.copy()
    df_plot[TARGET_COL] = y_s.values
    raw_synthetics_for_plot[method] = df_plot

real_plot = X_train.copy()
real_plot[TARGET_COL] = y_train.values

_ = plot_class_balance(real_plot, raw_synthetics_for_plot, cfg, save=True)
plt.show()
print("Class balance looks reasonable if synthetic proportions are close to real.")

## 4 · Real-Data Baseline Validation

We run `Evaluator.baseline(clf)` for all three classifiers on the real training split.
This step **validates the metric pipeline**: if these numbers look reasonable for a
dataset of this size and complexity, we can trust the TSTR deltas downstream.

Expected ballpark for the Aequitas education dataset with a balanced-accuracy target:
- Logistic Regression: BA ≈ 0.65–0.75
- XGBoost: BA ≈ 0.70–0.80
- MLP: BA ≈ 0.65–0.78

In [ ]:
evaluator = Evaluator(split, cfg)

baseline_rows = []
for clf_name in CLASSIFIERS:
    result = evaluator.baseline(clf_name)
    row = {
        "method"            : BASELINE,
        "classifier"        : clf_name,
        "balanced_accuracy" : result["balanced_accuracy"],
        "f1_macro"          : result["f1_macro"],
        "roc_auc"           : result["roc_auc"],
    }
    # Attach per-attribute fairness metrics
    for attr in PROTECTED_COLS:
        for metric in ("dpd", "eod", "di"):
            key = f"{attr}_{metric}"
            if key in result:
                row[key] = result[key]
    baseline_rows.append(row)
    print(f"  [{clf_name}]  BA={result['balanced_accuracy']:.4f}  "
          f"F1={result['f1_macro']:.4f}  AUC={result['roc_auc']:.4f}")

In [ ]:
baseline_df = pd.DataFrame(baseline_rows)

# Compute mean fairness metrics per classifier (across protected attrs)
for metric in ("dpd", "eod", "di"):
    cols = [c for c in baseline_df.columns if c.endswith(f"_{metric}")]
    if cols:
        baseline_df[f"mean_{metric}"] = baseline_df[cols].mean(axis=1)

display_cols = ["classifier", "balanced_accuracy", "f1_macro", "roc_auc",
                "mean_dpd", "mean_eod", "mean_di"]
display_cols = [c for c in display_cols if c in baseline_df.columns]

print("Real-data baseline metrics:")
print(baseline_df[display_cols].round(4).to_string(index=False))

In [ ]:
# Fairness check: flag any classifier that already violates thresholds on real data
print("\nFairness threshold checks on real data:")
print(f"  DPD > 0.1 is concerning (demographic parity violation)")
print(f"  EOD > 0.1 is concerning (equalized odds violation)")
print(f"  DI  < 0.8 is concerning (four-fifths rule violation)\n")

for _, row in baseline_df.iterrows():
    flags = []
    if "mean_dpd" in row and row["mean_dpd"] > 0.1:
        flags.append(f"mean_DPD={row['mean_dpd']:.3f}")
    if "mean_eod" in row and row["mean_eod"] > 0.1:
        flags.append(f"mean_EOD={row['mean_eod']:.3f}")
    if "mean_di"  in row and row["mean_di"]  < 0.8:
        flags.append(f"mean_DI={row['mean_di']:.3f}")
    status = "⚠️  " + ", ".join(flags) if flags else "✓ all within thresholds"
    print(f"  [{row['classifier']}]  {status}")

## 5 · TSTR Evaluation Loop

For each `(generator, classifier)` pair:
1. Call `evaluator.baseline(clf)` to set the correct real-data reference for that classifier
2. Train classifier on synthetic data
3. Test on the real held-out test set
4. Record utility metrics, fairness metrics, MMD, and correlation deltas

This is the core Train-on-Synthetic, Test-on-Real (TSTR) framework from the proposal.

In [ ]:
all_rows = baseline_rows.copy()  # seed with real baseline rows

for clf_name in CLASSIFIERS:
    # Re-run baseline for this classifier to anchor _real_metrics correctly
    evaluator.baseline(clf_name)

    for method, (X_synth, y_synth) in synthetic_data.items():
        logger.info(f"Evaluating [{method} | {clf_name}] ...")
        result = evaluator.evaluate(
            X_synth        = X_synth,
            y_synth        = y_synth,
            classifier_name= clf_name,
            generator_name = method,
            repetition     = 0,
            X_real_for_mmd = split.X_train,
        )
        # rename "generator" -> "method" for plotting compatibility
        row = {"method": result.pop("generator"), **result}
        all_rows.append(row)
        print(f"  [{method} | {clf_name}]  "
              f"BA={row['balanced_accuracy']:.4f}  "
              f"ΔBA={row['delta_balanced_accuracy']:+.4f}  "
              f"MMD={row['mmd']:.4f}")

print(f"\n{len(all_rows)} rows collected ({len(CLASSIFIERS)} baselines + "
      f"{len(synthetic_data) * len(CLASSIFIERS)} TSTR evaluations).")

In [ ]:
results_df = pd.DataFrame(all_rows)

# Add aggregated mean fairness columns (mean across protected attributes)
for metric in ("dpd", "eod", "di"):
    attr_cols = [c for c in results_df.columns
                 if c.endswith(f"_{metric}") and not c.startswith("fairness_gap")]
    if attr_cols:
        results_df[f"mean_{metric}"] = results_df[attr_cols].mean(axis=1)

# Save results
results_path = cfg["paths"]["results_dir"] / "goal3_results.csv"
results_path.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(results_path, index=False)
print(f"Results saved → {results_path}")
print(f"Shape: {results_df.shape}")
results_df.head(3)

## 6 · Utility Analysis

We examine how much predictive utility is lost when classifiers are trained on
synthetic data instead of real data.  
The primary metric is **balanced accuracy** (BA), which is robust to the class
imbalance present in the excellence-detection task. We also report F1-macro
and ROC-AUC for completeness.

In [ ]:
utility_cols = ["method", "classifier", "balanced_accuracy", "f1_macro", "roc_auc",
                "delta_balanced_accuracy", "delta_f1_macro", "delta_roc_auc", "mmd"]
utility_table = results_df[utility_cols].sort_values(
    ["classifier", "balanced_accuracy"], ascending=[True, False]
)
print("Utility metrics (all methods × classifiers):\n")
print(utility_table.round(4).to_string(index=False))

In [ ]:
# Best method per classifier by balanced accuracy
print("Best synthetic method per classifier (by balanced accuracy):\n")
synth_mask = results_df["method"] != BASELINE
best = (results_df[synth_mask]
        .groupby("classifier")
        .apply(lambda g: g.nlargest(1, "balanced_accuracy"))
        .reset_index(drop=True)
        [["classifier", "method", "balanced_accuracy", "delta_balanced_accuracy"]])
print(best.round(4).to_string(index=False))

In [ ]:
for metric in ["balanced_accuracy", "f1_macro", "roc_auc"]:
    fig = plot_utility_bar(results_df, metric, cfg, save=True)
    plt.show()

In [ ]:
fig = plot_mmd_bar(results_df, cfg, save=True)
plt.show()

print("\nMMD summary (lower = synthetic distribution closer to real):")
mmd_summary = (results_df[results_df["method"] != BASELINE]
               .drop_duplicates("method")[["method","mmd"]]
               .sort_values("mmd"))
print(mmd_summary.round(4).to_string(index=False))

## 7 · Fairness Analysis

For each protected attribute (`s_gender`, `f_ESCS_binary`) and each
`(method, classifier)` combination we measure:

| Metric | Definition | Good value |
|--------|-----------|------------|
| **DPD** | \|Pr[Ŷ=1\|A=0] − Pr[Ŷ=1\|A=1]\| | close to 0 |
| **EOD** | max(\|ΔTPR\|, \|ΔFPR\|) | close to 0 |
| **DI** | Pr[Ŷ=1\|A=0] / Pr[Ŷ=1\|A=1] | ≥ 0.8 (four-fifths rule) |

We compare real-trained models with synth-trained models. A good generator
**preserves** the fairness profile of the real baseline — it neither improves
nor worsens it artificially.

In [ ]:
fairness_base_cols = ["method", "classifier"]
dpd_cols = [c for c in results_df.columns if c.endswith("_dpd") and not c.startswith("fairness")]
eod_cols = [c for c in results_df.columns if c.endswith("_eod") and not c.startswith("fairness")]
di_cols  = [c for c in results_df.columns if c.endswith("_di")  and not c.startswith("fairness")]
mean_cols = [c for c in ["mean_dpd","mean_eod","mean_di"] if c in results_df.columns]
gap_cols  = [c for c in results_df.columns if c.startswith("fairness_gap_")]

fairness_table = results_df[fairness_base_cols + dpd_cols + eod_cols + di_cols + mean_cols].copy()
print("Fairness metrics (all methods × classifiers):\n")
print(fairness_table.round(4).to_string(index=False))

In [ ]:
# Per-attribute fairness bar charts
for metric in ["dpd", "eod", "di"]:
    if f"mean_{metric}" in results_df.columns:
        fig = plot_fairness_bar(results_df, f"mean_{metric}", cfg, save=True)
        plt.show()

In [ ]:
# Per-attribute breakdown (one chart per attr × metric)
for attr in PROTECTED_COLS:
    attr_rows = []
    for method in [BASELINE] + list(synthetic_data.keys()):
        sub = results_df[results_df["method"] == method]
        if sub.empty:
            continue
        for metric in ("dpd", "eod", "di"):
            col = f"{attr}_{metric}"
            if col in sub.columns:
                attr_rows.append({
                    "attribute": f"{method}",
                    metric: sub[col].mean(),   # mean across classifiers
                })
    if not attr_rows:
        continue
    attr_df = pd.DataFrame(attr_rows).groupby("attribute").mean().reset_index()

    for metric in ("dpd", "eod", "di"):
        if metric not in attr_df.columns:
            continue
        fig = plot_per_attribute_fairness(
            attr_df, metric, cfg,
            title=f"{metric.upper()} — {attr} (mean across classifiers)",
            filename_suffix=f"_{attr}_{metric}",
            save=True,
        )
        plt.show()

### Delta heatmaps

The heatmaps below show how each synthetic generator shifts utility and fairness
**relative to the real-data baseline**. Red = degradation relative to real; blue = improvement.

- For utility metrics (BA, F1, AUC): positive delta means the real model was better
  (i.e. synthetic training hurt). Red cells = utility loss.
- For fairness gap metrics: any non-zero value means the synthetic generator altered
  the fairness profile of the model.

In [ ]:
# Utility delta heatmap
delta_utility_cols = ["delta_balanced_accuracy", "delta_f1_macro", "delta_roc_auc"]
delta_utility_cols = [c for c in delta_utility_cols if c in results_df.columns]

if delta_utility_cols:
    fig = plot_delta_heatmap(
        results_df[results_df["method"] != BASELINE],
        delta_utility_cols,
        filename_suffix="utility",
        cfg=cfg,
        title="Utility delta: synthetic vs. real baseline (↑ red = worse)",
        save=True,
    )
    plt.show()

In [ ]:
# Fairness gap heatmap
gap_cols = [c for c in results_df.columns if c.startswith("fairness_gap_")]
if gap_cols:
    fig = plot_delta_heatmap(
        results_df[results_df["method"] != BASELINE],
        gap_cols,
        filename_suffix="fairness",
        cfg=cfg,
        title="Fairness gap: |synth – real| per protected attribute & metric",
        save=True,
    )
    plt.show()
else:
    print("No fairness_gap columns found — check that protected attrs were set correctly.")

In [ ]:
# Utility vs. fairness scatter (the key trade-off plot for the report)
if "mean_dpd" in results_df.columns:
    fig = plot_utility_fairness_scatter(
        results_df,
        utility_col  = "delta_balanced_accuracy",
        fairness_col = "mean_dpd",
        cfg          = cfg,
        save         = True,
    )
    plt.show()
    print("Points close to (0, 0) are Pareto-optimal: no utility loss, no fairness shift.")

## 8 · Summary

In [ ]:
print("=" * 70)
print("GOAL 3 SUMMARY")
print("=" * 70)

if not results_df.empty:
    synth = results_df[results_df["method"] != BASELINE]
    real  = results_df[results_df["method"] == BASELINE]

    mean_real_ba = real["balanced_accuracy"].mean()
    print(f"\nReal baseline (mean BA across classifiers): {mean_real_ba:.4f}")

    print("\nMean balanced accuracy per synthetic method (mean across classifiers):")
    tbl = synth.groupby("method")["balanced_accuracy"].mean().sort_values(ascending=False)
    for m, v in tbl.items():
        print(f"  {m:<20}  BA={v:.4f}  (Δ={v - mean_real_ba:+.4f})")

    if "mean_dpd" in synth.columns:
        print("\nMean DPD per synthetic method (mean across classifiers):")
        for m, v in synth.groupby("method")["mean_dpd"].mean().sort_values().items():
            print(f"  {m:<20}  mean_DPD={v:.4f}")

print("\nAll figures saved to:", cfg["paths"]["figures_dir"])
print("Results CSV saved to:", cfg["paths"]["results_dir"] / "goal3_results.csv")